# Assignment 1 & Assignment 2
__*Elise Wallace*__


## Goal of the Exercise

It is important to keep the overarching goal of this exercise, which is to calculate the safety stock by SKU, where the inventory type is a finished good (FG) and the stocking type is made-to-stock (MTS).

It is also important to notate the formula for calculating safety stock, which is below:

$$ SS = z_{\alpha} \times \sigma_{d} \times \sqrt{LT}$$

Knowing this, we know that the key features of this dataset would include the SKU, inventory type, stocking type, demand/quantity, and lead time.

## Importing Packages and Data

In [ ]:
import pandas as pd
import numpy as np
from plotnine import *
import scipy.stats as stats
from scipy.stats import norm

In [ ]:
transaction_data = pd.read_csv("../data/transaction_data.csv")
meta_data = pd.read_csv("../data/meta_data.csv")

## Question 1: DataFrame Formatting

As you load and inspect your transaction dataframe, you'll notice the column names don't follow a consistent convention. Make the necessary changes to ensure all column names follow the same convention as the metadata.

### Answer 1:

#### Headers for meta_data
We want to first get the headers for the meta data in order to identify what needs to be corrected or matched. We also want to get a description of the feature/header for our actual transactional dataset.

We want to display the metadata table first.

In [ ]:
meta_data

#### Meta_data Analysis

After looking at the table, we see that there are 9 features with the following data types:

| Feature | Data Type | 
| -------- | ---------- |
| sku_number | Qualitative Nominal |
| inventory_type | Qualitative Ordinal |
| stocking_type | Qualitative Nominal |
| lead_time | Quantitative Discrete |
| unit_price | Quantitative Continuous |
| manufacturing_site | Qualitative Nominal |
| division_code | Qualitative Nominal |
| transaction_date | Qualitative Ordinal |
| order_quantity | Quantitative Discrete |

Now, we'll check the transaction_data to see if the headers and the data types match.


In [ ]:
transaction_data.info()

#### Cross-Check & Creating Dictionary

Now we see that generally, all of the data type matches. The only one which we may need to perform some transformation on is "transaction_date".
As far as the headers, it looks like "lead_time", "division_code", and "order_quantity" need to be updated. But just in case, let's go ahead and rename the columns by overwriting the existing column names.

In [ ]:
transaction_title = ["sku_number", "inventory_type", "stocking_type", "lead_time", "unit_price", "manufacturing_site", "division_code", "transaction_date", "order_quantity"]

In [ ]:
transaction_data.columns = transaction_title

#### Validation for new column names

Next, we will look at the transaction data info to ensure that the correct headers were used.

In [ ]:
transaction_data.info()

## Question 2: 
Inspect the numeric features of your dataframe with the describe function. Provide a summary of these features, and in case there are any oddities/anomalies in your dataframe, locate and correct them.  Please include a note on any oddities you find and a rationale for your suggested correction. You can do so by including a text note in a corresponding markdown shell.

In [ ]:
transaction_data.describe()

### Answer 2:

We see that after running the describe function, there are negative numbers in the "unit_price" and "order_quantity". 
We also see a maximum lead time of 2800. We would want to also investigate any lead times which are beyond 3 standard deviations away from the mean.

#### Order Quantity less than 0

We see that we have 36,437 observations that have a negative order quantity. Looking at the inventory type, we see a mix of finished goods (FG), work-in-progress (WIP), and raw materials (RM). 

Typically, these types of transactions would indicate raw materials which were returned to the supplier, items in WIP which perhaps had a production order (but no manufacturing actually started) that was cancelled, or actual finished goods which were returned by the customer, or even identify a quality issue where items in WIP or even FGs had to be returned/canceled. It could also indicate excess materials which were scrapped.

Normally, we would keep these transactions as is, as it is common for materials to continuously move back and forth between the production lines. Since the ultimate goal of this entire exercise it to calculate safety stock, we need to look at the columns/variables that are used in the calculation of safety stock. This would include demand (quantity) and lead time. 

We performed further analysis below (describing the data for quantities less than 0, and for those greater than 0), and compared these against the entire dataset. We concluded that the mean, standard deviation, and lead time would not change by much.

Even if these transactions were returns or cancellations, the organization still manufactured the product and its replacements if a replacement was needed. Thus, we would consider **dropping** these values.

In [ ]:
transaction_data.query('order_quantity < 0')

In [ ]:
transaction_data.describe()

In [ ]:
transaction_data.query('order_quantity >= 0').describe()

In [ ]:
transaction_data_clean = transaction_data.query('order_quantity >= 0')

#### Order Quantity is 0
We also want to make sure that our data does not contain any order quantities where it is 0. Including these can often skew both the order quantity mean, and the lead time. If no actual order quantity was placed, we would consider this as an data entry error, or it could also indicate a cancellation. Rather than removing the transaction, it could be well-intended for the company's policy to still keep the transaction record, but update the quantity to 0. Nonetheless, for the purposes of our goal, it should be **omitted**.

In [ ]:
transaction_data_clean.query('order_quantity == 0').describe()

In [ ]:
transaction_data_clean = transaction_data_clean.query('order_quantity != 0')
transaction_data_clean.describe()

#### Unit Price less than 0

For the unit pricing, there could be a plethora of reasons as to why we see a negative unit price. They could be cost adjustments, supplier rebates, or it could also be a typo. Cost adjustments could be related to supplier-related cost adjustments, overhead/labor adjustments, or depending on the ERP system and the type of return, the transaction could appear as a negative unit price with a positive order quantity. Some organizations may want to see supplier rebates or cost adjustments differently than actual returns, to help with differentiating what type of "negative" inventory balance occurred. For example, at Amkor Technology, we tend to keep some transaction line items with a positive quantity but a negative unit price to indicate a credit that was issued to our customer for reconciliation purposes. Cost adjustments of raw materials are also transacted in a similar manner, where the quantity is left intact, but the unit price reflects a negative value to issue a credit to the supplier.

If more context is provided and these are actually errors, we would request corrections on these, or just take the mean or median of the unit price by SKU and inventory type.

However, we run into the same issues as the negative quantities. If the ultimate goal is to calculate the safety stock, any credits or cost adjustments should not be included in the order quantity. If these are truly rebates, reconciliations, or adjustments, we don't want the "duplicate" lead times to be included in our safety stock calculation. **In the grand scheme of things, we only have 10 transactions with a negative unit price. Thus, our suggestion is to REMOVE these transaction rows, as the order quantity and lead time values would not "move the needle" much.**

In [ ]:
transaction_data_clean.query('unit_price < 0').describe()

##### Removing Negative Unit Price

In [ ]:
transaction_data_clean = transaction_data_clean.query('unit_price >= 0')
transaction_data_clean.describe()

#### Unit price = 0

Next, we also want to look at anything with a unit price of 0. There is **no data** with a unit price of 0. No further action is needed.

In [ ]:
transaction_data_clean.query('unit_price == 0').describe()

#### Lead time greater than 3 standard deviations from the mean (outliers)

While the describe function from pandas already gave us the values for the mean and standard deviation for lead time, I don't want to manually write down and calculate the number to filter. Thus, I've also loaded numpy to perform the calculation, set 2 variables called "lt_3std" and "lt_mean" and using those variables to apply my filter.

In [ ]:
lt_3std = 3*np.std(transaction_data_clean["lead_time"])
lt_mean = np.mean(transaction_data_clean["lead_time"])

In [ ]:
transaction_data_clean.query('abs(lead_time - @lt_mean) > @lt_3std').describe()

##### Checking for any lead times with a "0" value

In [ ]:
transaction_data_clean.query('lead_time == 0').describe()

##### Lead Time Corrections

Based on the lead times that were in the transactional data, it is **extremely** unlikely that any lead time would be 3.5 to 7.7 years. Our intuition would tell us to *omit* or remove these line items, or to replace the lead time with the mean or median of the lead time of the sku_number and inventory type (omitting the outliers).

Considering that we only have 10 transactions where the lead time is greater than 1400 days, we would choose to remove these from our dataset.

*Note: The reason why we would not just look at the sku_number is because one SKU can have several inventory types, depending on how they are classified in each manufacturing process. As we add more "value" or "time" with each process, the value may change*

In [ ]:
transaction_data_clean = transaction_data_clean.query('abs(lead_time - @lt_mean) <= @lt_3std')
transaction_data_clean.describe()

## Question 3 — NaN Values
Your transactional dataset includes many missing values. Find and address (in your dataframe) the missing values by column. Again, remember to note the NaN values you found and provide a rationale for your suggested action.

### Answer 3:
We want to go through every column and sum the total numbers of NaN values in each column. Since we also REMOVED some rows from the previous step(s), we would also re-reun the isna().sum() function after removing to get the "corrected" values for the counts.

#### SKU Number
Since the SKU number is generally a **unique** identifier, we would NOT fill in the data. However, since we're only interested in calculating the safety stock by SKU and stocking type, we would ultimately opt to **remove** these rows.

However, it should be noted that it ultimately does not matter if we choose to keep these transaction rows as is, or replacing with a value of "Unknown". When we perform a calculation of the safety stock by SKU and inventory type, we would just have a safety stock calculation, but it does not carry any real meaning.

In [ ]:
transaction_data_clean.isna().sum()

##### Removing NaN values from SKU

In [ ]:
transaction_data_clean = transaction_data_clean[transaction_data_clean["sku_number"].notna()]
transaction_data_clean.describe()

#### Inventory Type
Inventory type has 12,511 transactions with missing values. 

However, since the ultimate goal of this exercise is to calculate safety stock by SKU for FG MTS products, there is no way for us to accurately "fill" this information in. We don't want to remove more items than necessary. We will later filter for any transactions with and inventory type of "FG", so we would ultimately leave the transactions alone.

If there is a future request to also calculate the safety stock based on inventory type, we would perform further clean-up measures to fill in the information, or omit the transactions from our dataset.

In [ ]:
transaction_data_clean.isna().sum()

#### Stocking Type
While stocking type is critical for us to filter and calculate safety stock by FG and MTS stocking types, we cannot accurately "fill" in the NaN values for stocking type, since each SKU can carry mutliple stocking types. Thus, we would opt to to **keep** the dataset as it, with a NaN value. We could also remove these rows, but the less that we touch the original dataset, the better.

If there was another request in the future to further cleanse the data, we would either opt to fill or remove these transaction rows.

In [ ]:
transaction_data_clean.isna().sum()

#### Lead Time
No NaN values for leadtime. No action needed.

#### Unit Price
No NaN values for unit price. No action needed.

#### Manufacturing Site
Again, this feature has no bearing in terms of calculating the safety stock by SKU and stocking type. Since the NaN values does contain important transactions that does carry weight/bearing on our safety stock calculation goal, we would choose to **keep** the transaction data, but **not** fill in the data.

In [ ]:
transaction_data_clean["manufacturing_site"].isna().sum()

In [ ]:
transaction_data_clean[transaction_data_clean["manufacturing_site"].isna()]

#### Division Code

Again, this feature has no bearing in terms of calculating the safety stock by SKU and stocking type. Since the NaN values does contain important transactions that does carry weight/bearing on our safety stock calculation goal, we would choose to **keep** the transaction data, but **not** fill in the data.

In [ ]:
transaction_data_clean["division_code"].isna().sum()

In [ ]:
transaction_data_clean[transaction_data_clean["division_code"].isna()]

#### Transaction Date
No NaN values for transaction date. No action needed.

In [ ]:
transaction_data_clean["transaction_date"].isna().sum()

#### Order Quantity
No NaN values for order quantity. No action needed.

In [ ]:
transaction_data_clean["order_quantity"].isna().sum()

#### Summary of NaN counts for each feature after filtering from Question 2

| Feature | Count of NaN | 
| -------- | ---------- |
| sku_number | 298 |
| inventory_type | 12,511 |
| stocking_type | 13,224 |
| lead_time | 0 |
| unit_price | 0 |
| manufacturing_site | 130,188 |
| division_code | 47,011 |
| transaction_date | 0 |
| order_quantity | 0 |

#### Summary of Decisions for each feature
For SKU number, since it is a unique identifier, we would not fill in the data. Considering the context of our goal to calculate the safety stock by SKU and stocking type, it ultimately does not matter if we keep or remove the rows. We would choose to just keep it for now.

For stocking type and inventory type, these features are needed in order to appropriately filter the data. However, there is no accurate way to "fill" the data in. Thus, we would just leave the rows alone, but we would **not** remove the transactions from the dataset. Our position is that the less that we "touch" the original data, the better.

For manufacturing site and divison code, we would keep the data as is, since this feature is not needed for calculating safety stock by SKU and stocking type. However, we would **NOT** remove the transactions from the dataset, since it does contain rows that have key information for our ultimate goal.

Lead time, unit price, transaction date, and order quantity would require no transformation, filling, or removing of NaN values, since there are **no** NaN values.


## Question 4 — Extracting The Country Values

To apply broader breakdowns of your transactional dataset, it will be helpful to add a country column. Recall that the metadata indicates the country information is somewhat captured in the site column. Create a new country column by using the manufacturing_site information. To do so, you will need the helper function str.startswith() (here is a [link](https://pandas.pydata.org/docs/reference/api/pandas.Series.str.startswith.html) to the official documentation), which is defined for a pandas Series object. As the documentation states, str.startswith() tests if the start of each string element matches a given pattern.

### Answer 4:
From the metadata, we know that the countries that we are interested in are USA, Canada, Poland, and Japan.

#### Find unique values under manufacturing site
Next, we want to create a list of the manufacturing sites. We see that there is a pattern, where PL = Poland, US = USA, CA = Canada, and JP = Japan. Rather than mapping out each unique value, we use this pattern to see if we can use the "start with" function. Ideally, we'd like to just see the values of the first 2 characters of the string to then map it to a dictionary.

In [ ]:
transaction_data_clean["manufacturing_site"].unique()

#### Creating a dictionary
In order to add a column based on the values of the manufacturing site, we need to create a dictionary first.

In [ ]:
country_map = {"CA" : "Canada",
               "JP" : "Japan",
               "PL" : "Poland",
               "US" : "USA"}

#### Mapping to new country
Creates a new column named "country", we take it from the manufacturing site, look at the string of the first 2 characters,and map it to our dictionary. We learned in class that the str function converts the selected column values into a string expression, and creating a index from ":2" would allow the system to select the manufacturing site, transform it into a string expression, and to grab the first 2 characters of the string. Since Python starts a 0, the index would stop **before** the 2nd character (character 0 and character 1). *(Stack Overflow, 2022)*

In [ ]:
transaction_data_clean["country"] = transaction_data_clean["manufacturing_site"].str[:2].map(country_map)

In [ ]:
transaction_data_clean.sample(10)

## Question 5 — Helpful Dimensional Information
It's time to focus on the subset of this dataset we need. Since we ultimately want to calculate safety stock, focus on finished goods designated as "make to stock." Create a filtered subset of your transaction dataset and write appropriate code to answer the following questions (for the filtered data).

+ How many unique SKUs are we working with?
+ How many unique Manufacturing Sites are we working with?
+ How many Divisions are we working with?
+ What are the top and bottom 10 transactions by order quantity?
+ What are the top and bottom 10 transactions by total sales value? Hint: Sales is not a feature in this dataset, so we must calculate it using the existing features.

### Answer 5:
#### Creating a filtered subset
We want to create a new subset called "MTS_data" where we are working with the transactions where the stocking_type is only "MTS" and the inventory_type is "FG".

In [ ]:
MTS_data = transaction_data_clean.query('stocking_type == "MTS" and inventory_type == "FG"')
MTS_data

#### Unique SKUs Count
To see the unique number of SKUs in our filtered subset, we want to do a value_counts. This value_counts will also automatically *ignore* NaN entries. We also want to find the LENGTH, so we use the len function.

We have 375 unique SKUs. Since we already "dropped" the NaN values, we only have 375 unique SKUs.

In [ ]:
len(MTS_data["sku_number"].value_counts())

#### Unique manufacturing sites
We perform the same exercise with the SKUs. There are 19 unique manufacturing sites, 20 if including NaN.

In [ ]:
len(MTS_data["manufacturing_site"].value_counts())

In [ ]:
len(MTS_data["manufacturing_site"].value_counts(dropna=False))

#### Unique divisions

There are 54 unique divisions for MTS transactions. 55 if including NaN.

In [ ]:
len(MTS_data["division_code"].value_counts())

In [ ]:
len(MTS_data["division_code"].value_counts(dropna=False))

#### Finding top 10 transactions by order quantity

In [ ]:
MTS_data.sort_values("order_quantity", ascending = False).head(10)

#### Finding bottom 10 transactions by order quantity
Since we also may want to see any non-negative transactions to exclude the credits or returns, we may want to opt to exclude looking at those.

In [ ]:
MTS_data.query('order_quantity > 0').sort_values("order_quantity").head(10)

#### Adding total sales value column
We add a new column titled "total_sales_value" and multiply the order quantity by the unit price.

In [ ]:
MTS_data["total_sales_value"] = MTS_data["order_quantity"]*MTS_data["unit_price"]

In [ ]:
MTS_data

#### Top 10 transactions by total sales value
Now that we've added a new column, we do the same sorting functions as above.

In [ ]:
MTS_data.sort_values("total_sales_value", ascending = False).head(10)

#### Bottom 10 transactions by total sales value

Although our team made the decision to include negative quantities, we thought it might be best to filter out any negative values, as typically we just want to see what is the lowest sales value. Since these possible indicate credits or returns, we also provided another table with the lowest sales values that are non-negative. Note that we already removed the transactions with a quantity of 0.

In [ ]:
MTS_data.sort_values("total_sales_value").head(10)

--- 
# Assignment 2 Continuation


## Question 1 — Required Transformation
Create a transformed dataset to filter for appropriate SKUs that are also aggregated with the following statistics: minimum, maximum, average, median, variance, and standard deviation of quantity, along with average lead time.

### Answer 1:
#### Data Cleansing
Similar to assignment 1, we want to clean the data by updating the headers to match the metadata, dropping any NaN values for SKU number, omit the transactions where we the order quantity is less than or equal to 0, omit any transactions where the unit price is less than or equal to 0, and remove any transactions with extreme outliers (3.5-7.7 years).

In this case, we only want the data to show where the stocking type is MTS. Since we already know from Assignment 1 that we would not fill in the NaN values, we can just filter for any MTS products. 

In [ ]:
lt_3std = 3*np.std(transaction_data["lead_time"])
lt_mean = np.mean(transaction_data["lead_time"])

transaction_data_clean = transaction_data.query('order_quantity >= 0 and unit_price >= 0 and abs(lead_time - @lt_mean) <= @lt_3std')
transaction_data_clean = transaction_data_clean.query('stocking_type == "MTS"')
transaction_data_clean = transaction_data_clean[transaction_data_clean["sku_number"].notna()]

transaction_data_clean.describe()

#### Data Transformation
In order to use aggregate the quantity by SKU, we have to create a new dataframe. We will use the "groupby" function, along with the aggregate function. Since we are trying to aggregate multiple summarizations of order_quantity, we will have to call the specific functions we'd like in brackets (Pandas development team, n.d.).

Because of the way we are aggregating the quantity and lead time by SKU, it creates multi-level headers. We will overwrite this by just manually renaming the column headers.

##### Aggregating Order Quantities and Lead Times

In [ ]:
transformed_data = (transaction_data_clean
    .groupby("sku_number")
    .aggregate({"order_quantity":["min", "max", "mean", "median", "var", "std"],
                "lead_time":"mean"})
    .reset_index())

transformed_data

##### Overwriting Column Names

In [ ]:
transformed_data.columns = ["sku_number",
                           "min_order_quantity",
                           "max_order_quantity",
                           "avg_order_quantity",
                           "median_order_quantity",
                           "var_order_quantity",
                           "std_order_quantity",
                           "avg_lead_time"]
transformed_data.sample(5)

## Question 2 — Safety Stock Calculation
Use the transformed data from Question 1 to calculate three safety stock values for each SKU for the service level coefficients of 75%, 90%, and 95%. You will need to find the z-score of the given service level coefficients, for which you can use the [ppf](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.rv_continuous.ppf.html) function of SciPy.Stat module of the [SciPy](https://docs.scipy.org/doc/scipy/reference/) package, or simply use the numeric values from any standard normal distribution function.

### Answer 2:

#### Type of Distribution
##### Using Coefficient of Variation Rule for Normal Distribution
First, we want to check to see if our inventory is normally distributed or if a different distribution should be used. We will add another column for the coefficient of variation. For formula for the coefficient of variation is as follows:

$\text{CV} = \frac{\text{Std Dev}}{\text{Mean}}$

If the CV is less thna 30%, demand is relatively stable and predictable, relative to its volume, indicating that we can most likely use a normal distribution (Oracle, n.d.).

Now, we calculate the CV and find the SKUs which we can apply a normal distribution.

In [ ]:
transformed_data["cv"] = transformed_data["std_order_quantity"]/transformed_data["avg_order_quantity"]

In [ ]:
transformed_data.query('cv <= 0.3')

##### Variance to Mean Ratio
Next, we will use a variance to mean ratio to determine whether we the SKUs have a binomial, poisson, or negative binomial distribution, using different "generalized" thresholds (CASRAI, n.d.).

$\text{VMR} = \frac{\text{Variance}}{\text{Mean}}$

In [ ]:
transformed_data["VMR"] = transformed_data["var_order_quantity"]/transformed_data["avg_order_quantity"]

##### Filtering for Binomial Distributions
After filtering for any SKUs with a high coefficient of variation and variance to mean ratio much then 1, we identified 0 SKUs which indicate a binomial distribution.

In [ ]:
transformed_data.query('cv > 0.3 and VMR < 0.8')

##### Filtering for Poisson Distributions
After filtering for any SKUs with a high coefficient of variation and medium variance to mean ratio close to 1, we identified 0 SKUs which indicate a poisson distribution. We selected a ratio between 0.8 and 1.2

In [ ]:
transformed_data.query('cv > 0.3 and VMR >= 0.8 and VMR <1.2')

##### Filtering for Negative Binomial Distributions
After filtering for any SKUs with a high coefficient of variation and a variance to mean ratio greater than 1, we identified the remaining non-normally distributed SKUs as having a negative binomial distribution. This meeans that most SKUs have a fairly low demand/order quantity but experience erratic or unpredictable high-quantity spikes. Using a safety stock calculation which assumes a normal distribution, we may experience more stockouts on these SKUs.

In [ ]:
transformed_data.query('cv > 0.3 and VMR >= 1.2')

#### Find z-scores for Percentiles for Normal Distributions

In [ ]:
z_75 = norm.ppf(0.75)
z_90 = norm.ppf(0.90)
z_95 = norm.ppf(0.95)

#### Calculating Safety Stock for Normal Distributions
Now we use the following function to calculate the safety stock for each SKU. We will use our transformed dataset to complete this.

$$ SS = z_{\alpha} \times \sigma_{d} \times \sqrt{LT}$$

We also want to display whole numbers for these, so we will use the floor division (//) and remainder (%) functions to achieve this. The floor division will give us the "rounded" integer. We also use the remainder function, along with a logical operator to return a value of 0 or 1. If the remainder > 0, then the result should be 1. If the remainder is 0, then the result of that subcalculation would be 0. We also want to change the data type to "int64".

To demonstrate this, we've provided an example below:

- We select a random value of 17.4
- When we do the floor division, it will return a value of 17.
- When we do the remainder, it should return a value of 4.
- We also have a logical operator for the remainder, saying if the remainder (4) is greater than 0, then return a value of 1.
- In this case, since we have a remainder that is greater than 0, it would return a value of 1.
- We add this boolean response of 1 to the floor division result of 17.
- Our final result would be 18.

##### Safety Stock at 75%

In [ ]:
raw_ss_75_norm = z_75 * transformed_data["std_order_quantity"] * np.sqrt(transformed_data["avg_lead_time"])
transformed_data["ss_75_norm"] = (raw_ss_75_norm // 1 + (raw_ss_75_norm % 1 > 0)).astype("int64")
transformed_data.sample(5)

##### Safety Stock at 90%

In [ ]:
raw_ss_90_norm = z_90 * transformed_data["std_order_quantity"] * np.sqrt(transformed_data["avg_lead_time"])
transformed_data["ss_90_norm"] = (raw_ss_90_norm // 1 + (raw_ss_90_norm % 1 > 0)).astype("int64")
transformed_data.sample(5)

##### Safety Stock at 95%

In [ ]:
raw_ss_95_norm = z_95 * transformed_data["std_order_quantity"] * np.sqrt(transformed_data["avg_lead_time"])
transformed_data["ss_95_norm"] = (raw_ss_95_norm // 1 + (raw_ss_95_norm % 1 > 0)).astype("int64")
transformed_data.sample(5)

#### Calculating Safety Stock for Negative Binomial Distributions
For the SKUs with a negative binomial distribution, we want to identify the reorder point and average demand during lead time to calculate safety stock. The formula is as follows:
$$\text{Safety Stock} = ROP - \mu_{LT}$$

To obtain the reorder point, we would use the SciPy.stats nbinom.ppf function.
To use the function, we would also need to calculate several other parameters. We would need the shape/dispersion (n), probability parameter (p), average demand during lead time, and average variance during lead time. The formulas are as follows:

$$n = \frac{\mu_{LT} \times p}{1 - p}$$
$$p = \frac{\mu_{LT}}{\sigma^2_{LT}}$$
$$\mu_{LT} = \mu_{\text{daily}} \times \text{Lead Time}$$
$$\sigma^2_{LT} = \sigma^2_{\text{daily}} \times \text{Lead Time}$$

The n parameter (sometimes known as k) and the p-parameter are from a statistical technique called "method of moments" (Zaiontz, n.d.).

In [ ]:
transformed_data["mean_during_lt"] = transformed_data["avg_order_quantity"]*transformed_data["avg_lead_time"]
transformed_data["var_during_lt"] = transformed_data["var_order_quantity"]*transformed_data["avg_lead_time"]

In [ ]:
transformed_data["p_parameter"] = transformed_data["mean_during_lt"] / transformed_data["var_during_lt"]

In [ ]:
transformed_data["n_parameter"] = ((transformed_data["mean_during_lt"] * 
                                    transformed_data["p_parameter"]) / 
                                    (1- transformed_data["p_parameter"]))

In [ ]:
transformed_data["rop_75"] = stats.nbinom.ppf(0.75, transformed_data["n_parameter"], transformed_data["p_parameter"])
transformed_data["rop_90"] = stats.nbinom.ppf(0.90, transformed_data["n_parameter"], transformed_data["p_parameter"])
transformed_data["rop_95"] = stats.nbinom.ppf(0.95, transformed_data["n_parameter"], transformed_data["p_parameter"])

##### Adding Safety Stock columns for negative binomial distributions
Now, we substrate the reorder point from the average demand during lead time to obtain the safety stock at 75%, 90%, and 95%.

We would use the same rounding methodology as the normal distribution safety stocks to ensure that we are providing whole values for safety stock quantities.

In [ ]:
raw_ss_75_bn = transformed_data["rop_75"] - transformed_data["mean_during_lt"]
raw_ss_90_bn = transformed_data["rop_90"] - transformed_data["mean_during_lt"]
raw_ss_95_bn = transformed_data["rop_95"] - transformed_data["mean_during_lt"]

transformed_data["ss_75_bn"] = (raw_ss_75_bn // 1 + (raw_ss_75_bn % 1 > 0)).astype("int64")
transformed_data["ss_90_bn"] = (raw_ss_90_bn // 1 + (raw_ss_90_bn % 1 > 0)).astype("int64")
transformed_data["ss_95_bn"] = (raw_ss_95_bn // 1 + (raw_ss_95_bn % 1 > 0)).astype("int64")

#### Finalized Safety Stock Column
Since we want to just consolidate the "correct" safety stock to each SKU, we will use the np.where function to call upon the safety stock at 75^, 90%, and 95%, depending on the value of the coefficient of variation. Since we already identified that anything with a coefficient of variation which is greater than 30% is using a negative binomial distribution, we just need to build logic that says:

If the CV is less than or equal to 0.3, then grab the NORMAL safety stock, else use the binomial negative safety stock calculation.

In [ ]:
transformed_data["ss_75"] = np.where(transformed_data["cv"] <= 0.3,
                                     transformed_data["ss_75_norm"],
                                     transformed_data["ss_75_bn"])

transformed_data["ss_90"] = np.where(transformed_data["cv"] <= 0.3,
                                     transformed_data["ss_90_norm"],
                                     transformed_data["ss_90_bn"])

transformed_data["ss_95"] = np.where(transformed_data["cv"] <= 0.3,
                                     transformed_data["ss_95_norm"],
                                     transformed_data["ss_95_bn"])

transformed_data.sample(5)

## Question 3 — Safety Stock Distribution
Now that we have the safety stock values, write appropriate code to answer the following questions. For this question, you may focus on the 95% service level coefficient.

- Which SKU has the largest safety stock?
- Which SKU has the smallest safety stock?
- What is the average safety stock value across all SKUs?

### Answer 3:
Before answering questions, now I want to create a copy of my transformed data into a neat and clean table with just the SKU and safety stocks for each method (since this would also be a way to validate the data). We create a new dataframe from the aggregated transformed data with sku number, and the 3 columns for safety stock at 95% with both methods (normal and negative binomial).

In [ ]:
selected_columns = ["sku_number", "ss_95_norm", "ss_95_bn", "ss_95"]
data_95_ss = transformed_data[selected_columns].copy()

#### Largest Safety Stock
In order to find the SKU with the largest safety stock, using a 95% service level agreement (SLA), we would need to sort the values by DESCENDING values from the "ss_95" column. We also used the .head function, but wanted to print multiples, just in case there were multiple SKUs with the highest safety stock quantity.

The SKU with the largest safety stock is AC3ADABF.

In [ ]:
data_95_ss.sort_values("ss_95", ascending = False).head(5)

#### Smallest Safety Stock
In order to find the SKU with the smallest safety stock, using a 95% service level agreement (SLA), we would need to sort the values by ASCENDING values from the "ss_95" column. We also used the .head function, but wanted to print multiples, just in case there were multiple SKUs with the smallest safety stock quantity.

The SKU with the smallest safety stock is 58F77D5E.

In [ ]:
data_95_ss.sort_values("ss_95", ascending = True).head(5)

#### Average Safety Stock
To obtain the average safety stock, we can just use the .describe() function to find the mean.

The average safety stock is about ~122 units.

In [ ]:
data_95_ss.describe()

## References

CASRAI. (n.d.). Negative binomial regression overdispersion. CASRAI Guides. Retrieved November 12, 2026, from https://casrai.org/guides/negative-binomial-regression-overdispersion

Oracle. (n.d.). NetSuite help center: Inventory management guidelines. Oracle Help Center. Retrieved November 12, 2026, from https://docs.oracle.com/en/cloud/saas/netsuite/ns-online-help/article_0702084957.html#subsect_75201401147

Pandas development team. (n.d.). Pandas.DataFrame.aggregate. pandas. Retrieved November 7, 2026, from https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.aggregate.html

Stack Overflow. (2022, March 16). How to substring values from a pandas df [Online forum post]. Stack Exchange Inc. https://stackoverflow.com/questions/71537795/how-to-substring-values-from-a-pandas-df

Zaiontz, C. (n.d.). Method of moments: Negative binomial distribution. Real Statistics Using Excel. Retrieved November 12, 2026, from https://real-statistics.com/distribution-fitting/method-of-moments/method-of-moments-negative-binomial-distribution/